<a id='Index'></a>
# Handlers index:
* <a href='#Definitions'>Definitions</a>
* <a href='#Classes-for-Data-Manipulation'>Classes for Data Manipulation</a>
* <a href='#Classes-for-Visualisation-of-manipulated-data'>Classes for Visualisation of manipulated data</a>
* <a href='#Linear-Regression-Pipeline'>Linear Regression Pipeline</a>
* <a href='#Random-Forest-Pipeline'>Random Forest Pipeline</a>
* <a href='#Visualisation-exercises-03-04'>Visualisation exercises 03-04</a>

<a href='#Index'>Handlers Index</a>
<a id='Definitions'></a>

# Definitions

# Save your local username

In [ ]:
import os
import pwd
username_local = pwd.getpwuid( os.getuid() ).pw_name

# Define all directory names (for local data, plot, ...)

In [ ]:
# Workspace directory:
homedir=os.environ.get('MYWS')

# IO directories:
local_data_dir    = os.path.join(homedir,'NB_Dataframes')
local_data_dir_ro = os.path.join(homedir,'NB_Dataframes_read_only')
local_ml_dir      = os.path.join(homedir,'NB_ML_models')
local_ml_dir_ro   = os.path.join(homedir,'ML_models_read_only')

# Plot directories:
local_plot_general_dir        =  os.path.join(homedir,'NB_Plot')
local_manipulation_plot_dir   =  os.path.join(local_plot_general_dir,'ManipulationPlot')
local_regression_plot_dir     =  os.path.join(local_plot_general_dir,'RegressionPlot')
local_classification_plot_dir =  os.path.join(local_plot_general_dir,'ClassificationPlot')

# Directory for the source data:
basedir =  os.path.join(homedir,'sbahn_data')

# LustrePrefix
LustrePrefix = 'file://'

# Make sure all directories exist or create them:
if (not os.path.isdir(homedir)):
    os.mkdir(homedir)
if (not os.path.isdir(local_data_dir)):
    os.mkdir(local_data_dir)
if (not os.path.isdir(local_ml_dir)):
    os.mkdir(local_ml_dir)
if (not os.path.isdir(local_plot_general_dir)):
    os.mkdir(local_plot_general_dir)
if (not os.path.isdir(local_manipulation_plot_dir)):
    os.mkdir(local_manipulation_plot_dir)
if (not os.path.isdir(local_regression_plot_dir)):
    os.mkdir(local_regression_plot_dir)
if (not os.path.isdir(local_classification_plot_dir)):
    os.mkdir(local_classification_plot_dir)    

# Seed and weights for splitting the dataset
* We use a random split to separate into training and test dataset
* The **weights** define the amount of data in each set (%)
* We can fix the **seed** of the random split to reproduce the results

In [ ]:
dfSeed_list  = [4561767182015543883]
#dfWeights = [[0.9, 0.1]] # 90% training, 10% test
dfWeights = [[0.5, 0.5]]  # 50% training, 50% test

# Which seed we are using for this run (default 0):
dfSeed_index = 0

# Labels (procedure in case of multiple split, implemented only in script)
dfWeights_float = 1.0
for i in range(len(dfWeights)):
    dfWeights_float = dfWeights[i][0]*dfWeights_float
dfWeights_int = int(round(dfWeights_float,2)*100)
dfWeights_label = str(dfWeights_int)
dfWeights_test_int = int(round(dfWeights[0][1],2)*100)
dfWeights_test_label = str(dfWeights_test_int)

# Suffix for all IO data to specify seed and weight:
suffix_s_w = '_s_'+str(dfSeed_index)+'_w_'+dfWeights_label + '_wtest_'+dfWeights_test_label

# Import all requirements

In [ ]:
import os

import time

# Originally in the Manipulation Noteboook

import math, random
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from math import sqrt

# Originally in Handlers

from pyspark.sql.functions import col, countDistinct, when, udf, asc, desc, dayofmonth, \
    from_unixtime, month,dayofmonth, minute, hour, unix_timestamp, year, date_format, mean
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType
from pyspark.sql.types import *
from datetime import datetime


from pyspark.sql import DataFrame

from pyspark.sql import Window
import pyspark.sql.functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import Normalizer, VectorAssembler, StringIndexer
from pyspark.ml.regression import \
    LinearRegression, \
    GeneralizedLinearRegression, \
    DecisionTreeRegressor, \
    RandomForestRegressor, \
    GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from pyspark.ml.classification import LinearSVC
from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import countDistinct


import numpy as np
import pandas as pd
print("Pandas version: ", pd.__version__)
import matplotlib.pyplot as plt

from numpy import concatenate
from pandas import DataFrame
from pandas import concat

import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

import matplotlib.ticker as ticker

<a href='#Index'>Handlers Index</a>
<a id='Classes-for-Data-Manipulation'></a>

# Class for handling Dataframes in Manipulation

In [ ]:
class DataHandler:
    def convertDateToUnix(datestring):
        date_format_1 = "%d.%m.%Y"
        date_format_2 = "%d.%m.%Y %H:%M:%S"
        date_format_3 = "%Y-%m-%d %H:%M:%S" 

        date_formats = (date_format_1, date_format_2, date_format_3)
        for fmt in date_formats:
            try: 
                return int(datetime.strptime(datestring, fmt).strftime('%s'))
            except ValueError:
                pass
        raise ValueError('No valid date format found!')
        
    def calcTimeDiff(tsmp_ist, tsmp_soll):
        td = tsmp_ist- tsmp_soll
        return int(round(td.total_seconds()/60))
    
    def minuteRange(minute):
        if minute < 30:
            return 0
        else:
            return 30
        
    def reshapeDf(data_set, steps = 1):
        data_set = data_set.orderBy(asc('SERVICE_ID'), asc('MONAT'), asc('TAG'), asc('STUNDE_SOLL'), asc('MINUTE_SOLL'))
        #import pyspark.sql.functions as F
        df_temp = data_set.withColumn('id', F.monotonically_increasing_id())
        str = 'VERSPAETUNG'

        #lag: prev, lead: next
        for i in range(steps):
            str = 'prev{0}_'.format(i) + str
            df_reshaped = df_temp.select(
                '*',
                *([F.lag(F.col(c), default = 0).over(Window.orderBy('id').partitionBy('SERVICE_ID')).alias('prev{0}_'.format(i) + c) for c in df_temp.columns])
            )

            #10= Start from Startstation(?)
            df_reshaped = df_reshaped.withColumn(str, when(df_reshaped.ZUGEREIGNIS_TYP == 10, 0).otherwise(df_reshaped[str]))

            cols_drop = []
            for colu in data_set.columns:
                cols_drop.append('prev{0}_'.format(i) + 'id')
                if 'VERSPAETUNG' not in colu:
                    cols_drop.append('prev{0}_'.format(i) + colu)


            df_temp = df_reshaped.drop(*cols_drop)
        cols_drop = ['id']
        str = 'id'
        for i in range(steps):
            str = 'prev{0}_'.format(i) + str
            cols_drop.append('prev{0}_'.format(i) + 'id')
            if i > 0:
                cols_drop.append('prev{0}_'.format(i) + 'VERSPAETUNG')
                cols_drop.append(str)
        df_temp = df_temp.drop(*cols_drop)
        return df_temp
    
    def dfSplit(data_set, split_id): #split_id: index according to which we are splitting
        list_id = data_set.select(split_id).distinct().rdd.flatMap(lambda x: x).collect()
        df_array = [data_set.where(col(split_id) == x)for x in list_id]

        return df_array, list_id
    
udfConvertDateToUnix = udf(DataHandler.convertDateToUnix, IntegerType())
udfCalcTimeDiff = udf(DataHandler.calcTimeDiff)
udfMinuteRange = udf(DataHandler.minuteRange, IntegerType())

<a href='#Index'>Handlers Index</a>
<a id='Classes-for-Visualisation-of-manipulated-data'></a>

# Class for Visualisation of manipulated data

In [ ]:
class DataVisualize:

    # Create of a DataFrame with statistical information on each line:
    def calcStats(dataset_pd):
        #Groupby: A groupby operation involves some combination of splitting the object, applying a function,
        #and combining the results. This can be used to group large amounts of data and compute operations on these groups.

        #Groups the Delays (in mins) by Line Number, then applies get_stats (below) for statistics.

        df_ = dataset_pd['VERSPAETUNG'].groupby(dataset_pd['ZUGEREIGNIS_LINIE']).apply(DataVisualize.get_stats).unstack()
        df_ = df_.sort_values('count')
        return df_
    
    def meanDelay(dataset_pd):
        df_DS100_mean = pd.DataFrame(pd.Series(dataset_pd['ZUGEREIGNIS_DS100'].unique()))
        df_DS100_mean.set_index(0, drop = True, inplace = True)
        print(df_DS100_mean_delay[0:10])
        #This is now an empty dataframe with names of the stations as index

        df_stats = DataVisualize.calcStats(dataset_pd)

        for carrier in df_stats.index.values:  #df_stats.index.values = numbers of lines (see above)
            df1 = dataset_pd[dataset_pd['ZUGEREIGNIS_LINIE'] == carrier] #filter data for ONE line
            #See above: Group the delays by STATION. (we are working on a single line now), then apply statistics as above.
            test = df1['VERSPAETUNG'].groupby(dataset_pd['ZUGEREIGNIS_DS100']).apply(DataVisualize.get_stats).unstack()
            #Each df in the array contains info on a line, stations + mean delay 
            df_DS100_mean[carrier] = test.loc[:, 'mean'] 
        df_DS100_mean.columns = ['S' + str(col) for col in df_DS100_mean.columns]
        return df_DS100_mean
   
    def heatMap(dataset_pd, flag):
        mask = dataset_pd.isnull()
        # The default cmap is sns.cm.rocket. To reverse it set cmap to sns.cm.rocket_r
        # The flag turns the colorbar on/off
        sns.heatmap(dataset_pd, linewidth = 0.01, mask = mask, cbar = flag, vmax = 4, vmin = 0, cmap = sns.cm.rocket_r)
    
    def get_stats(group):
        return {'min': group.min(), 'max': group.max(),
                'count': group.count(), 'mean': group.mean()}

    # Barplot: "height" should be displayed as .2f: TODO!!!
    def autolabel(rects):
        """Attach a text label above each bar in *rects*, displaying its height."""
        for rect in rects:
                height = rect.get_height()
                ax.annotate('{}'.format(height),
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

In [ ]:
class DataVisualize_ext:

    def convert_to_pandas(spark_df):
        """
        This function will safely convert a spark DataFrame to pandas.
        """
        # Iterate over columns and convert each timestamp column to a string
        timestamp_cols = []
        for column in spark_df.schema:
            if column.dataType == TimestampType():
                   # Append column header to list
                timestamp_cols.append(column.name)
                   # Set column to string using date_format function
                spark_df = spark_df.withColumn(column.name, date_format(column.name, "yyyy-MM-dd HH:mm:ss"))
           # Convert to a pandas DataFrame and reset timestamp columns
        pandas_df = spark_df.toPandas()
        for column_header in timestamp_cols:
            pandas_df[column_header] = pandas_df[column_header].astype("datetime64[ns]")
        return pandas_df

<a href='#Index'>Handlers Index</a>
<a id='Linear-Regression-Pipeline'></a>

# Class for ML Linear Regression DO NOT EXECUTE

In [ ]:
# Some references:
# https://spark.apache.org/docs/2.2.0/ml-pipeline.html
# PHB pp. 343-350: Scikit-Learn pipeline (analogous to Spark)
# PHB pp. 360-362: Scikit-Learn cross validation (analogous to Spark)
# PHB pp. 363-375: Best model

class MLPipeline_LR:
    def linReg(data_train, cols_to_inx, cols_inx):
        
        # Build the pipeline
        
        # StringIndexer:
        # https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html?highlight=stringindexer#pyspark.ml.feature.StringIndexer   
        # StringIndexer maps a string column of labels to an ML column of label indices.
        indexers = [StringIndexer(inputCol=column, outputCol=column+"_INDEX", handleInvalid = 'keep') \
                    for column in cols_to_inx]

        # VectorAssembler: 
        # A feature transformer that merges multiple columns into a vector column.
        # https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html?highlight=vectorassembler#pyspark.ml.feature.VectorAssembler
        assembler = VectorAssembler(inputCols = cols_inx,outputCol = "feature")

        # Normaliser:
        # Normalise a vector to have unit norm using the given p-norm.
        normalizer = Normalizer(inputCol = 'feature', outputCol = 'features_uni', p = 1.0) 
        
        # LinearRegression 
        # Default:
        # class pyspark.ml.regression.LinearRegression(featuresCol='features', labelCol='label', predictionCol='prediction',
        # maxIter=100, regParam=0.0, elasticNetParam=0.0, tol=1e-06, fitIntercept=True, standardization=True, solver='auto',
        # weightCol=None, aggregationDepth=2, loss='squaredError', epsilon=1.35)
        # LABEL = target (= what we want to predict)
        # LinearRegression is the choice of the architecture, 
        # ... LinearRegression is NOT the application of the model to the data, which happens below (estimator).
        algorithm = LinearRegression( featuresCol = normalizer.getOutputCol(), labelCol = 'VERSPAETUNG', \
                                     maxIter=100, regParam=0.001)
        # Some alternatives:
        ##algorithm = GeneralizedLinearRegression(family="gaussian", link="identity", maxIter=10, regParam=0.3, featuresCol=normalizer.getOutputCol(), labelCol='VERSPAETUNG')
        ##algorithm = DecisionTreeRegressor(featuresCol=normalizer.getOutputCol(), labelCol='VERSPAETUNG')
        ##algorithm = RandomForestRegressor(featuresCol=normalizer.getOutputCol(), labelCol='VERSPAETUNG')
        ##algorithm = GBTRegressor(featuresCol=normalizer.getOutputCol(), labelCol='VERSPAETUNG', maxIter=10)
        
        # This is the ESTIMATOR which can be applied to a DataFrame and fitted to deliver a weighted model.
        pipeline = Pipeline(stages = indexers + [assembler, normalizer, algorithm])

        # Define the model EVALUATOR as RMSE:
        modelEvaluator=RegressionEvaluator(labelCol="VERSPAETUNG", predictionCol="prediction", metricName="rmse")
        
        # VALIDATION:
        # Grid of parameters (Grid Search: see PHB pp. 373-374): 
        # - Two regularisation parameters are defined (see e.g. https://spark.apache.org/docs/1.5.2/ml-linear-methods.html )
        # - The model is run for each combination of the parameters.
        paramGrid = ParamGridBuilder().addGrid(algorithm.regParam, [0.1, 0.01]).addGrid(algorithm.elasticNetParam, [0, 1]).build()

        # CROSS VALIDATION:
        # Training Data are split into numFolds subsets.
        # Each subset is used in turn as a validation set, while the rest is acting as training set.
        crossval = CrossValidator(estimator=pipeline,
                              estimatorParamMaps=paramGrid,
                              evaluator=modelEvaluator, # contains information about the label and the prediction columns (cf. linRegTest below)
                              numFolds=10) # Min. 2!
        
        #Fit the model to the training data:
        cvModel = crossval.fit(data_train)
        
        # Return the evaluator and the weighted model
        return modelEvaluator, cvModel
 
        # Make an INFERENCE on the TEST dataset, given the evaluator and the model:
    def linRegTest(data_test_prediction, model_evaluator, model):

        # Do the INFERENCE via transform:
        prediction_prediction = model.transform(data_test_prediction)

        # Include a scatter plot (...work in progress...)
        selected = prediction_prediction.select('VERSPAETUNG','prediction')
        selected_pandas = selected.toPandas()
        plt.clf()
        plt.close()
        plt.scatter(selected_pandas.prediction, selected_pandas.VERSPAETUNG)
        plt.show()
        
        # EVALUATION of the inference:
        rmse_res = model_evaluator.evaluate(prediction_prediction)        
        
        print("Root Mean Squared Error (RMSE) on data = %g" % rmse_res)

        return rmse_res

<a href='#Index'>Handlers Index</a>
<a id='Random-Forest-Pipeline'></a>

# Class for ML Classification (Random Forest)

In [ ]:
# https://spark.apache.org/docs/2.2.0/ml-pipeline.html
# PHB pp. 343-350: Scikit-Learn pipeline (analogous to Spark)
# PHB pp. 360-362: Scikit-Learn cross validation (analogous to Spark)
# PHB pp. 363-375: Best model

class MLPipeline_RF:
    def clfcTrain(data_train_classification, cols_to_inx, cols_inx):
        #Build the pipeline

        # StringIndexer:
        # ...as in regression...
        indexers = [StringIndexer(inputCol=column, outputCol=column+"_INDEX", handleInvalid = 'keep') \
                    for column in cols_to_inx]

        # VectorAssembler: 
        # ...as in regression...
        assembler = VectorAssembler(inputCols = cols_inx,outputCol = "feature") 

        # Normaliser:
        # ...as in regression...
        normalizer = Normalizer(p = 1.0, inputCol = 'feature', outputCol = 'features_uni')
        
        # RandomForest 
        # ...as in regression... + 
        # Parameters for RF are numTrees (default: 20) and maxDepth (default: 5).
        # https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.RandomForestClassifier.html?highlight=randomforestclassifier#pyspark.ml.classification.RandomForestClassifier
        algorithm = RandomForestClassifier(labelCol = 'classification', featuresCol = normalizer.getOutputCol(),\
                                           numTrees = 100)

        # This is the ESTIMATOR (...as in regression...).
        pipeline = Pipeline(stages = indexers + [assembler, normalizer, algorithm])

        #Define the model EVALUATOR as accuracy:
        modelEvaluator = MulticlassClassificationEvaluator(labelCol="classification", \
                                                           predictionCol="prediction", metricName = 'accuracy')

        # VALIDATION
        # We do not define a parameter grid here (empty).
        paramGrid = ParamGridBuilder().build()

        # CROSS VALIDATION
        # ...as in regression...
        crossval = CrossValidator(estimator=pipeline,
                              estimatorParamMaps=paramGrid,
                              evaluator=modelEvaluator, # contains information about the label and the prediction columns (cf. clfcLoopTest below)
                              numFolds=10) # Use 10 (min. 2!)

        #Fit the model to the training data:
        model = crossval.fit(data_train_classification)
        
        # Return the evaluator and the weighted model
        return modelEvaluator, model
    
        # Make an INFERENCE on the TEST dataset, given the evaluator and the model:
        # Evaluates an _array_ of DataFrames and stores each accuracy in a list.
    def clfcLoopTest(data_test_array, model_evaluator, model):
        accu_arr = []
 
        # Loop over DataFrames (one for each S-Bahn STATION)
        for i in range(len(data_test_array)):
        
        # Do the INFERENCE via transform:
            pred = model.transform(data_test_array[i])
            
        # EVALUATION of the inference:
            accuracy = model_evaluator.evaluate(pred)
            
        # Append the accuracy for the data at each station:
            accu_arr.append(accuracy)
        return accu_arr

<a href='#Index'>Handlers Index</a>
<a id='Visualisation-exercises-03-04'></a>

# Class for visualisation of Classification results    

In [1]:
class PredictionVisualize:

    # Get the geographical coordinates of the stations contained in the DataFrame df_pd_stations.
    # - list_names_seq are ALL ORDERED stations of the S1 line
    # - list_ds are the stations in the order of the accuracy DataFrame
    # - abbr_stations is the dictionary of station names
    # - accur_arr is the list of accuracies of all stations (one list for each of the 5 models)
    # - threshold contains the accuracy limits ( not acceptable / acceptable / precise)
    
    def getCoordsNoInt(list_names_seq, df_pd_stations, accur_arr, abbr_stations, list_ds, threshold):

        coords = []
        colors = []

        # From the Pandas DataFrame, create lists corresponding to:
        # - the DS100 codes of the stations:
        list_exc_stations = df_pd_stations['Abk.'].to_list()
        # - the coordinates of the stations:
        list_exc_lat = df_pd_stations['Latitude'].to_list()
        list_exc_lon = df_pd_stations['Longitude'].to_list()
        
        # If the station is associated to an accuracy,
        # we assign a color to the station in the S1 line according to the delay accuracy threshold ... 
        for i in range(len(list_names_seq)):
            if(list_names_seq[i] in list_ds):
                
                # EXERCISE 3
                # - In both list_ds and list_exc_stations, get the INDEX of the station we are considering in the for loop.
                # - Use the function "index":
                x = list_ds.index(list_names_seq[i])
                y = list_exc_stations.index(list_names_seq[i])
                
                if  np.abs(accur_arr[x]) >= threshold[0]:
                    colors += ['green']
                elif np.abs(accur_arr[x]) >= threshold[1]:
                    colors += ['orange']
                else:
                    colors += ['black']
                
                # - Add the geographical coordinates of this station to the list of all coordinates:
                coords += [[list_exc_lat[y], list_exc_lon[y]]]
                
                # - Print the name of the station
                print(abbr_stations[list_names_seq[i]])
                # - Print the last element in the list of coordinates
                print(coords[-1])
                
        return coords, colors    

    # Get the geographical coordinates of the stations with geopy.
    def getCoords(list_names_seq, accur_arr, abbr_stations, list_ds, threshold):
        # Nominatim is the geocoder for OpenStreetMap data
        geolocator = Nominatim(user_agent = 'SBahn Delay Prediction') 
        coords = []
        colors = []

        # If the station is associated to an accuracy,
        # we assign a color to the station in the S1 line according to the delay accuracy threshold ... 
        for i in range(len(list_names_seq)):
            if(list_names_seq[i] in list_ds):

                # EXERCISE 4 (optional)
                # - In list_ds, get the INDEX of the station we are considering in the for loop.
                # - Use the function "index":
                x = list_ds.index(list_names_seq[i])
                if  np.abs(accur_arr[x]) >= threshold[0]:
                    colors += ['green']
                elif np.abs(accur_arr[x]) >= threshold[1]:
                    colors += ['orange']
                else:
                    colors += ['black']
            
                # 'Altbach' is recognised by the geolocator only as 'Altbach Bahnhof'
                # When abbr_stations[list_names_seq[i]] is 'Altbach',
                # it must be replaced with 'Altbach Bahnhof'
                # Add 1 or 2 lines so that abbr_bahnhofs is correct also for this stop. 
                # You can use a conditional statement, or replace, ...
                # __________________________________________________
                # __________________________________________________

                # - Select the name of the station we are considering for the geolocation
                location = geolocator.geocode(abbr_stations[abbr_bahnhofs])                
                # Add the geographical coordinates of this station to the list of all coordinates:
                coords += [[location.latitude, location.longitude]]
                
                # - Print the name of the station
                print(abbr_stations[list_names_seq[i]])
                # - Print the last element in the list of coordinates
                print(coords[-1])

        return coords, colors    
    
    # Draw stations on the map with their full name (as a pop-up) instead of the DS100 code:
    # - popups_all ist the list of ALL ORDERED S1 stations.
    # - popups_actual are the stations in the order of the accuracy DataFrame.
    def drawMapDic(coords, colors, popups_all, popups_actual, dict_popups):
        map_object = folium.Map(location=[48.783066, 9.180544], zoom_start=13)
        x=0
        for i in range(len(colors)):
            while True:
                if (popups_all[x] in popups_actual):
                    x=x+1
                    break
                else:
                    x=x+1
            marker = folium.features.Marker(coords[i], icon=folium.Icon(color=colors[i]), popup=dict_popups[popups_all[x-1]])
            map_object.add_child(marker)
        folium.PolyLine(coords, color = 'blue', weight = 2.5, opacity=1).add_to(map_object)
        return map_object